# run_hydro_plots notebook

This notebook embeds the full code from `run_hydro_plots.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Run examples (PowerShell):
python run_hydro_plots.py --mode single --basin_id SE000197
python run_hydro_plots.py --mode zoom_multi
python run_hydro_plots.py --mode zoom_multi --zoom_start_date 2007-01-01 --zoom_end_date 2009-12-31 --basins BEWA0120,BEWA0040
python run_hydro_plots.py --mode gothenburg_two_months
python run_hydro_plots.py --mode single --basin_id SE000197 --plot_start_date 2006-01-01 --plot_end_date 2009-12-31

Interactive HTML (pan/zoom in browser):
python run_hydro_plots.py --mode single --basin_id SE000197 --interactive
python run_hydro_plots.py --mode zoom_multi --interactive --zoom_start_date 2006-01-01 --zoom_end_date 2009-12-31
"""

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from calibrate_validate_dpwm import DBFReader
from calibration_io import basin_hydrograph_title
from dpwm_model import DPWM
from hydrograph_interactive import build_overlay_hydrograph, write_interactive_html


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Single, configurable script for hydrograph plotting workflows.")
    parser.add_argument("--mode", choices=["single", "zoom_multi", "gothenburg_two_months"], default="single")
    parser.add_argument("--root", default=".")
    parser.add_argument("--basin_id", default="SE000197")
    parser.add_argument("--basins", default="CH000127,DESN1585,FR002785,GB000215,SE000197,ES000700")
    parser.add_argument("--best_params_csv", default="Results/calibration/calibration_validation_results.csv")
    parser.add_argument("--best_params_sep", default=",")
    parser.add_argument("--rain_csv", default="Results/forcing/rainplusmelt.csv")
    parser.add_argument("--pet_csv", default="Results/forcing/PET.csv")
    parser.add_argument("--streamflow_csv", default="data/streamflow/estreams_timeseries_streamflow_v02.csv")
    parser.add_argument("--params_dbf", default="data/parameters/Parameters_basins.dbf")
    parser.add_argument("--area_field", default="area_estre")
    parser.add_argument("--q_obs_unit", choices=["m3s", "mmday"], default="m3s")
    parser.add_argument("--warm_span_days", type=int, default=365)
    parser.add_argument("--window_days", type=int, default=4018)
    parser.add_argument("--plot_days", type=int, default=700)
    parser.add_argument("--plot_start_date", default="")
    parser.add_argument("--plot_end_date", default="")
    parser.add_argument(
        "--zoom_start_date",
        default="2006-01-01",
        help="For mode=zoom_multi: first day of hydrograph window (default 2006-01-01).",
    )
    parser.add_argument(
        "--zoom_end_date",
        default="2009-12-31",
        help="For mode=zoom_multi: last day of hydrograph window (default 2009-12-31).",
    )
    parser.add_argument("--include_warmup", action="store_true")
    parser.add_argument("--annotate_invq", action="store_true")
    parser.add_argument("--out_dir", default="Results/hydrographs_merged")
    parser.add_argument(
        "--interactive",
        action="store_true",
        help="Write zoomable HTML (open in browser) instead of PNG.",
    )
    parser.add_argument(
        "--also_png",
        action="store_true",
        help="With --interactive, also save the static PNG.",
    )
    return parser.parse_known_args()[0]


def kge_metrics(q_obs: np.ndarray, q_sim: np.ndarray) -> tuple[float, float, float, float]:
    q_obs = np.asarray(q_obs, dtype=float)
    q_sim = np.asarray(q_sim, dtype=float)
    valid = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs = q_obs[valid]
    q_sim = q_sim[valid]
    if q_obs.size < 2:
        return np.nan, np.nan, np.nan, np.nan
    std_obs = np.std(q_obs)
    mean_obs = np.mean(q_obs)
    if std_obs == 0 or mean_obs == 0:
        return np.nan, np.nan, np.nan, np.nan
    r = np.corrcoef(q_obs, q_sim)[0, 1]
    alpha = np.std(q_sim) / std_obs
    beta = np.mean(q_sim) / mean_obs
    kge = 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)
    return kge, r, alpha, beta


def kge_inverse_flow(q_obs: np.ndarray, q_sim: np.ndarray, eps: float = 1e-6) -> float:
    valid = np.isfinite(q_obs) & np.isfinite(q_sim) & (q_obs > eps) & (q_sim > eps)
    if np.sum(valid) < 2:
        return np.nan
    inv_obs = 1.0 / q_obs[valid]
    inv_sim = 1.0 / q_sim[valid]
    return kge_metrics(inv_obs, inv_sim)[0]


def plot_hydrograph(
    basin_id: str,
    df: pd.DataFrame,
    out_png: Path,
    title: str,
    overlay_precip: bool = False,
    show_metrics: bool = True,
    q_obs_markers_only: bool = False,
    *,
    interactive: bool = False,
    also_png: bool = False,
) -> None:
    if interactive:
        out_html = out_png.with_suffix(".html")
        fig = build_overlay_hydrograph(
            df,
            title,
            q_obs_markers_only=q_obs_markers_only,
        )
        write_interactive_html(fig, out_html)
        print(f"Saved: {out_html} (open in a web browser to pan/zoom)")
        if not also_png:
            return
    q_obs = pd.to_numeric(df["q_obs"], errors="coerce").to_numpy(dtype=float)
    q_sim = pd.to_numeric(df["q_sim"], errors="coerce").to_numpy(dtype=float)
    precip = pd.to_numeric(df["precip"], errors="coerce").to_numpy(dtype=float)
    dates = pd.to_datetime(df["date"])

    kge, r, alpha, beta = kge_metrics(q_obs, q_sim)
    kge_inv = kge_inverse_flow(q_obs, q_sim)
    metrics_text = f"KGE(Q)={kge:.3f} | KGE(1/Q)={kge_inv:.3f} | r={r:.3f} | alpha={alpha:.3f} | beta={beta:.3f}"

    if overlay_precip:
        fig, ax_q = plt.subplots(1, 1, figsize=(12, 5.8))
        ax_p = ax_q.twinx()
        ax_p.bar(dates, precip, color="#bcdffb", width=1.0, alpha=0.7, label="Precipitation", zorder=1)
        ax_p.set_ylabel("P (mm d⁻¹)")
        ax_p.invert_yaxis()
        ax_q.set_zorder(2)
        ax_q.patch.set_alpha(0.0)
        if q_obs_markers_only:
            ax_q.plot(dates, q_obs, color="black", linestyle="None", marker="x", markersize=2.0, label="Observed Q")
        else:
            ax_q.plot(dates, q_obs, color="#8ec5ff", linewidth=1.6, linestyle="--", label="Observed Q")
        ax_q.plot(dates, q_sim, color="#0b2c85", linewidth=1.8, linestyle="-", label="Simulated Q")
        ax_q.set_title(title)
        ax_q.set_ylabel("Q (mm d⁻¹)")
        ax_q.set_xlabel("Date")
        ax_q.grid(alpha=0.25)
        h1, l1 = ax_q.get_legend_handles_labels()
        h2, l2 = ax_p.get_legend_handles_labels()
        leg = ax_q.legend(h1 + h2, l1 + l2, loc="upper right", framealpha=0.95)
        leg.set_zorder(1000)
        if show_metrics:
            ax_q.text(0.99, 0.98, metrics_text, transform=ax_q.transAxes, ha="right", va="top", fontsize=9)
    else:
        # Same overlay layout as zoom_multi (inverted P on secondary axis).
        fig, ax_q = plt.subplots(1, 1, figsize=(12, 5.8))
        ax_p = ax_q.twinx()
        ax_p.bar(dates, precip, color="#bcdffb", width=1.0, alpha=0.75, align="center", label="Precipitation", zorder=1)
        ax_p.set_ylabel("P (mm d⁻¹)")
        ax_p.invert_yaxis()
        ax_q.set_zorder(2)
        ax_q.patch.set_alpha(0.0)
        if q_obs_markers_only:
            ax_q.plot(dates, q_obs, color="black", linestyle="None", marker="x", markersize=2.0, label="Observed Q", zorder=3)
        else:
            ax_q.plot(dates, q_obs, color="#8ec5ff", linewidth=1.6, linestyle="--", label="Observed Q", zorder=3)
        ax_q.plot(dates, q_sim, color="#0b2c85", linewidth=1.8, linestyle="-", label="Simulated Q", zorder=3)
        ax_q.set_title(title)
        ax_q.set_ylabel("Q (mm d⁻¹)")
        ax_q.set_xlabel("Date")
        ax_q.grid(alpha=0.25)
        h1, l1 = ax_q.get_legend_handles_labels()
        h2, l2 = ax_p.get_legend_handles_labels()
        ax_q.legend(h1 + h2, l1 + l2, loc="upper right", framealpha=0.95)
        if show_metrics:
            ax_q.text(0.99, 0.98, metrics_text, transform=ax_q.transAxes, ha="right", va="top", fontsize=9)

    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)
    print(f"Saved: {out_png}")


def load_series_inputs(args: argparse.Namespace) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    rain_df = pd.read_csv(Path(args.root) / args.rain_csv)
    pet_df = pd.read_csv(Path(args.root) / args.pet_csv)
    dates = pd.to_datetime(rain_df["date"])
    return rain_df, pet_df, dates


def load_best_row(args: argparse.Namespace, basin_id: str) -> pd.Series:
    best_df = pd.read_csv(Path(args.root) / args.best_params_csv, sep=args.best_params_sep)
    row = best_df.loc[best_df["basin_id"] == basin_id]
    if row.empty:
        raise ValueError(f"basin_id {basin_id!r} not found in {args.best_params_csv}")
    return row.iloc[0]


def simulate_q(args: argparse.Namespace, basin_id: str, row: pd.Series, rain_df: pd.DataFrame, pet_df: pd.DataFrame):
    params = [float(row["alpha1"]), float(row["Sm"]), float(row["alpha2"]), float(row["Kf"]), float(row["Ks"])]
    dates = pd.to_datetime(rain_df["date"])
    precip = rain_df[basin_id].to_numpy(dtype=float)
    pet = pet_df[basin_id].to_numpy(dtype=float)
    model = DPWM(params, warm_span_days=args.warm_span_days)
    flux_output, _, _ = model.simulate(precip, pet)
    return dates, precip, flux_output.Q


def load_q_obs_mmday(args: argparse.Namespace, basin_id: str, row: pd.Series) -> pd.DataFrame:
    q_obs_df = pd.read_csv(Path(args.root) / args.streamflow_csv, usecols=["dates", basin_id]).rename(
        columns={"dates": "date", basin_id: "q_obs"}
    )
    q_obs_df["date"] = pd.to_datetime(q_obs_df["date"])
    q_obs_df["q_obs"] = pd.to_numeric(q_obs_df["q_obs"], errors="coerce")
    if args.q_obs_unit == "m3s":
        if "area_km2" in row.index and np.isfinite(float(row["area_km2"])):
            area_km2 = float(row["area_km2"])
        else:
            dbf = DBFReader(str(Path(args.root) / args.params_dbf))
            area_rec = dbf.read_records(["basin_id", args.area_field], where_basin_ids=[basin_id]).get(basin_id)
            if area_rec is None:
                raise ValueError(f"Could not find basin_id={basin_id} in {args.params_dbf}")
            area_km2 = float(area_rec[args.area_field])
        q_obs_df["q_obs"] = q_obs_df["q_obs"] * 86.4 / area_km2
    return q_obs_df


def run_single(args: argparse.Namespace) -> None:
    rain_df, pet_df, _ = load_series_inputs(args)
    row = load_best_row(args, args.basin_id)
    dates, precip, q_sim = simulate_q(args, args.basin_id, row, rain_df, pet_df)
    q_obs_df = load_q_obs_mmday(args, args.basin_id, row)

    plot_df = pd.DataFrame({"date": dates, "precip": precip, "q_sim": q_sim}).merge(q_obs_df, how="left", on="date")

    if not args.include_warmup:
        plot_df = plot_df.iloc[int(args.warm_span_days) :].copy()

    if args.plot_start_date.strip():
        plot_df = plot_df.loc[plot_df["date"] >= pd.to_datetime(args.plot_start_date)]
    if args.plot_end_date.strip():
        plot_df = plot_df.loc[plot_df["date"] <= pd.to_datetime(args.plot_end_date)]
    if not args.plot_start_date.strip() and not args.plot_end_date.strip():
        plot_df = plot_df.iloc[: int(args.plot_days)].copy()

    if plot_df.empty:
        raise ValueError("No data left after filters.")

    title = basin_hydrograph_title(args.basin_id)
    if args.annotate_invq and "KGE_invQ_cal" in row.index and "KGE_invQ_val" in row.index:
        title += f"\nKGE_invQ_cal={float(row['KGE_invQ_cal']):.3f} | KGE_invQ_val={float(row['KGE_invQ_val']):.3f}"

    out_png = Path(args.root) / args.out_dir / f"hydrograph_{args.basin_id}_single.png"
    plot_hydrograph(
        args.basin_id,
        plot_df,
        out_png,
        title,
        overlay_precip=True,
        show_metrics=True,
        interactive=args.interactive,
        also_png=args.also_png,
    )


def run_zoom_multi(args: argparse.Namespace) -> None:
    basins = [b.strip() for b in args.basins.split(",") if b.strip()]
    start_date = pd.Timestamp(args.zoom_start_date)
    end_date = pd.Timestamp(args.zoom_end_date)
    rain_df, pet_df, _ = load_series_inputs(args)

    for basin_id in basins:
        row = load_best_row(args, basin_id)
        dates, precip, q_sim = simulate_q(args, basin_id, row, rain_df, pet_df)
        q_obs_df = load_q_obs_mmday(args, basin_id, row)
        plot_df = pd.DataFrame({"date": dates, "precip": precip, "q_sim": q_sim}).merge(q_obs_df, how="left", on="date")
        plot_df = plot_df.iloc[int(args.warm_span_days) :].copy()
        plot_df = plot_df.loc[(plot_df["date"] >= start_date) & (plot_df["date"] <= end_date)].copy()
        if plot_df.empty:
            print(f"Skipped {basin_id}: no data in zoom window {start_date.date()}–{end_date.date()}")
            continue

        title = basin_hydrograph_title(
            basin_id,
            date_start=f"{start_date.date()}",
            date_end=f"{end_date.date()}",
        )
        if "KGE_invQ_cal" in row.index and "KGE_invQ_val" in row.index:
            title += f"\nKGE_invQ_cal={float(row['KGE_invQ_cal']):.3f} | KGE_invQ_val={float(row['KGE_invQ_val']):.3f}"
        tag = f"{start_date:%Y%m%d}_{end_date:%Y%m%d}"
        out_png = Path(args.root) / args.out_dir / f"hydrograph_{basin_id}_zoom_{tag}.png"
        # Keep zoom_multi layout consistent with gothenburg_two_months:
        # precipitation overlay (inverted axis) + Q panel style with marker-only observations.
        plot_hydrograph(
            basin_id,
            plot_df,
            out_png,
            title,
            overlay_precip=True,
            show_metrics=False,
            q_obs_markers_only=True,
            interactive=args.interactive,
            also_png=args.also_png,
        )


def run_gothenburg_two_months(args: argparse.Namespace) -> None:
    basin_id = "SE000197"
    title = basin_hydrograph_title(basin_id)
    rain_df, pet_df, _ = load_series_inputs(args)
    row = load_best_row(args, basin_id)
    dates, precip, q_sim = simulate_q(args, basin_id, row, rain_df, pet_df)
    q_obs_df = load_q_obs_mmday(args, basin_id, row)
    df = pd.DataFrame({"date": dates, "precip": precip, "q_sim": q_sim}).merge(q_obs_df, how="left", on="date")

    windows = [("2006-07-01", "2006-08-31", "gothenburg_2006_07_08"), ("2008-01-01", "2008-02-29", "gothenburg_2008_01_02")]
    for start, end, tag in windows:
        sub = df.loc[(df["date"] >= pd.Timestamp(start)) & (df["date"] <= pd.Timestamp(end))].copy()
        if sub.empty:
            print(f"No data for window {start} to {end}")
            continue
        out_png = Path(args.root) / args.out_dir / f"hydrograph_{tag}.png"
        plot_hydrograph(
            basin_id,
            sub,
            out_png,
            title,
            overlay_precip=True,
            show_metrics=False,
            q_obs_markers_only=True,
            interactive=args.interactive,
            also_png=args.also_png,
        )


def main() -> None:
    args = parse_args()
    if args.mode == "single":
        run_single(args)
    elif args.mode == "zoom_multi":
        run_zoom_multi(args)
    else:
        run_gothenburg_two_months(args)


# Execute the script entry point
main()
